# 01b · Caso de negocio — ¿el estudiante aprueba (≥ 70) o pierde el curso (< 70)?

> Segundo notebook del caso, basado en `notebooks/01_caso_negocio.ipynb`. Dataset: `data/student_exam_performance.csv` (100.000 estudiantes × 44 columnas). Diccionario: `data/README.md`.
> **Regla de aprobación de este caso: el estudiante aprueba con una nota de 70 o más; con menos de 70 pierde el curso.**

> ⚠️ **El dataset trae su propia columna `pass_status`, calculada con corte en 50.** Aquí **no se usa**: el objetivo de este caso se **deriva de `exam_score` con el corte en 70**.
> Los notebooks `02b` y `03b`, `src/` y `tests/` **ya usan el corte de 70** (objetivo `pass_70`). Los `.md` del repositorio (`reports/`, `especificaciones/`, `README.md`, `CLAUDE.md`, `data/README.md`…) **todavía mencionan el corte de 50 y `pass_status`** y hay que actualizarlos para que coincidan con este notebook.

Este notebook responde tres cosas, **con números**: qué significa «perder el curso», cuánto cuesta equivocarse y **qué métricas dicen si el modelo sirve (y cómo leerlas)**.
Su resumen va a `reports/00_caso.md`.

**Regla de oro:** `exam_score` solo se lee aquí para *describir* el objetivo. **Nunca es una variable de entrada del modelo** (de ella sale la respuesta).

## 1. La pregunta y la regla de aprobación

**Pregunta de negocio:** ¿qué estudiantes **no alcanzarán 70** en el examen y perderán el curso, para ofrecerles apoyo **antes** de que lo presenten?

| Resultado | Regla sobre `exam_score` (0 a 100) | En el código |
|---|---|---|
| **Pass** (aprueba) | nota **≥ 70** | `fail = False` |
| **Fail** (pierde el curso) | nota **< 70** | `fail = True` |

- **Clase de interés (positiva): `Fail`.** «Positivo» no quiere decir «bueno»: quiere decir *la clase que queremos detectar*.
- **Quién decide:** el/la coordinador(a) académico(a), que usa la alerta para asignar tutoría o refuerzo.
- **Cuándo se decide:** antes del examen. Solo se pueden usar variables que ya se conocen en ese momento.
- **Metodología de costos:** lo más costoso es que **el estudiante pierda el curso sin que nadie lo haya detectado**.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PASS_THRESHOLD = 70        # el estudiante aprueba con 70 o más; con menos de 70 pierde el curso

csv_path = next((p for p in [Path("../data/student_exam_performance.csv"), Path("data/student_exam_performance.csv"),
                              Path("../data/raw/student_exam_performance.csv"), Path("data/raw/student_exam_performance.csv")]
                 if p.exists()), None)
assert csv_path is not None, "No encontré data/student_exam_performance.csv"

# Se leen SOLO las columnas necesarias. exam_score NO se usará como variable de entrada.
# 'previous_exam_score' se lee únicamente para el ejemplo ilustrativo de la sección 4 (no es el modelo).
obj = pd.read_csv(csv_path, usecols=["exam_score", "pass_status", "previous_exam_score"])
n = len(obj)

fail = (obj.exam_score < PASS_THRESHOLD).to_numpy()          # True = pierde el curso (clase de interés)
assert not obj.exam_score.isna().any(), "hay notas vacías"

conteo = pd.Series({"Pass (≥ 70)": int((~fail).sum()), "Fail (< 70)": int(fail.sum())})
display(pd.DataFrame({"estudiantes": conteo, "proporción": (conteo / n).round(4)}))
print(f"Tasa de Fail con el corte en {PASS_THRESHOLD}: {fail.mean():.1%}   (con el corte de 50 del dataset original era {(obj.exam_score < 50).mean():.1%})")

# ¿Cuántos estudiantes cambian de resultado al subir el corte de 50 a 70?
cambia = pd.crosstab(obj.pass_status.rename("con corte 50 (pass_status)"), pd.Series(np.where(fail, "Fail", "Pass"), name="con corte 70"))
display(cambia)
print(f"Aprueban con 50 pero pierden el curso con 70: {int(((obj.exam_score >= 50) & (obj.exam_score < 70)).sum()):,} estudiantes "
      f"({((obj.exam_score >= 50) & (obj.exam_score < 70)).mean():.1%} del total)")

# ¿Qué tan cerca del corte (70) está la gente?
filas = []
for margen in [2, 5, 10]:
    cerca = (obj.exam_score - PASS_THRESHOLD).abs() < margen
    filas.append({"distancia al corte": f"menos de {margen} puntos", "estudiantes": int(cerca.sum()),
                  "% del total": round(cerca.mean() * 100, 1), "% de ellos que pierde el curso": round(fail[cerca.to_numpy()].mean() * 100, 1)})
display(pd.DataFrame(filas))

**Lo que cambia al subir el corte a 70 (hallazgos del caso):**

- **`Fail` pasa a ser la clase mayoritaria:** **69,2 %** de los estudiantes (69.210) no llega a 70, frente a 22,6 % con el corte de 50. Solo aprueba **30,8 %**.
- **46,6 % de los estudiantes (46.599) aprobaría con 50 y pierde el curso con 70**: el mismo examen, un estándar más exigente.
- **Casi cara o cruz cerca del corte:** **21,8 %** de los estudiantes está a menos de 5 puntos de 70 y de ellos **53,7 %** pierde el curso. Ningún modelo será perfecto en esa franja.

**Qué consecuencias tiene que reprobar sea lo habitual:**

1. **«Avisar a todos» pasa a ser una estrategia fuerte**, porque se equivoca solo con el 30,8 % que aprobaría (se ve en la sección 3).
2. **El valor del modelo está en identificar con seguridad a quienes SÍ aprobarán**, para no gastar tutoría en ellos y concentrarla en el resto.
3. **Una sola métrica no basta**: el *recall* de `Fail` se «gana» avisando a todos, y la exactitud se «gana» diciendo siempre `Fail` (ver sección 4).
4. **La capacidad de tutoría importa**: no se puede apoyar al 69 % de los estudiantes; por eso hay que **ordenar por riesgo** (métricas por capacidad, sección 4).

## 2. Equivocarse no cuesta lo mismo

| | El estudiante **pierde el curso** (`Fail`, nota < 70) | El estudiante **aprueba** (`Pass`, nota ≥ 70) |
|---|---|---|
| **El modelo avisa** («en riesgo») | **Acierto** (verdadero positivo, VP): recibe apoyo a tiempo | **Falso positivo (FP)**: una tutoría que no hacía falta |
| **El modelo no avisa** («tranquilo») | **Falso negativo (FN)**: pierde el curso y nadie lo detectó | **Acierto** (verdadero negativo, VN) |

**Error más caro: el falso negativo.** Un estudiante que pierde el curso sin que nadie lo haya detectado pierde tiempo, dinero y avance académico, y el apoyo ya no llega a tiempo.
**Error menos caro: el falso positivo.** Se gasta una tutoría que quizá no hacía falta (y aun así puede ayudar al estudiante).

**Proporción: 5 a 1 como supuesto de trabajo** (perder el curso cuesta 5 veces una tutoría innecesaria). **No sale del dataset**: la institución debe estimarla. La sección 3 muestra qué pasa con otras proporciones.
**Alerta sobre este supuesto:** como reprueba el 69 %, la institución no podrá dar tutoría a todos los que avise; **TODO** estimar el costo real de una tutoría y la capacidad disponible.

In [ ]:
CLASSES = ["Pass", "Fail"]          # índice 0 = Pass (≥ 70), índice 1 = Fail (< 70, clase de interés)

FALSE_POSITIVE_COST = 1             # avisar a quien habría aprobado           (supuesto)
FALSE_NEGATIVE_COST = 5             # NO avisar a quien pierde el curso         (supuesto)

# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Diagonal = 0.
COST_MATRIX = [[0, FALSE_POSITIVE_COST],
               [FALSE_NEGATIVE_COST, 0]]

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
display(cost)

## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Hay **tres estrategias sin modelo**:

| Estrategia | Qué hace | Se equivoca con… |
|---|---|---|
| **Nunca avisar** (siempre `Pass`) | No ofrece apoyo a nadie | Todos los que pierden el curso (**FN**, el error caro) |
| **Avisar a todos** (siempre `Fail`) | Ofrece apoyo a todos | Los que habrían aprobado (**FP**, el error barato) |
| **Azar según proporciones** | Avisa al azar, al 69,2 % de los estudiantes | Un poco de cada |

In [ ]:
def business_metrics(y_fail, alert, cost_matrix=COST_MATRIX):
    """Todas las métricas de negocio del caso, a partir de quién pierde el curso y a quién avisó el modelo.

    y_fail: True si el estudiante realmente pierde el curso (nota < 70).
    alert:  True si el modelo avisa que está en riesgo.
    """
    y_fail = np.asarray(y_fail, dtype=bool)
    alert = np.asarray(alert, dtype=bool)
    vp = int((y_fail & alert).sum())      # perdía el curso y se avisó
    fn = int((y_fail & ~alert).sum())     # perdía el curso y NO se avisó  (error caro)
    fp = int((~y_fail & alert).sum())     # habría aprobado y se avisó      (error barato)
    vn = int((~y_fail & ~alert).sum())    # habría aprobado y no se avisó
    total = len(y_fail)

    recall = vp / (vp + fn) if (vp + fn) else 0.0
    precision = vp / (vp + fp) if (vp + fp) else 0.0
    npv = vn / (vn + fn) if (vn + fn) else 0.0            # de los «tranquilos», ¿cuántos aprueban de verdad?
    specificity = vn / (vn + fp) if (vn + fp) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if (4 * precision + recall) else 0.0
    cm = np.array(cost_matrix, dtype=float)               # [real][predicho], índice 0 = Pass, 1 = Fail
    costo = (cm[0, 0] * vn + cm[0, 1] * fp + cm[1, 0] * fn + cm[1, 1] * vp) / total

    return {"costo por estudiante": costo, "recall Fail": recall, "precisión Fail": precision,
            "NPV (confianza al tranquilizar)": npv, "especificidad": specificity, "F1": f1, "F2": f2,
            "exactitud balanceada": (recall + specificity) / 2, "exactitud": (vp + vn) / total,
            "alertas por 100": 100 * (vp + fp) / total, "VP": vp, "FN": fn, "FP": fp, "VN": vn}


rng = np.random.default_rng(42)
estrategias = {
    "Nunca avisar (siempre Pass)": np.zeros(n, dtype=bool),
    "Avisar a todos (siempre Fail)": np.ones(n, dtype=bool),
    "Azar según proporciones": rng.random(n) < fail.mean(),
}
base = pd.DataFrame({k: business_metrics(fail, v) for k, v in estrategias.items()}).T
cols = ["costo por estudiante", "recall Fail", "precisión Fail", "NPV (confianza al tranquilizar)", "exactitud", "alertas por 100"]
display(base[cols].round(3))

tasa_fail = float(fail.mean())
p_pass = 1 - tasa_fail
mejor_costo_base = float(base["costo por estudiante"].min())
print(f"Mejor costo sin modelo: {mejor_costo_base:.2f} por estudiante ({base['costo por estudiante'].idxmin()}) | Tasa base de Fail: {tasa_fail:.1%}")

In [ ]:
# ¿Cambia la conclusión si el costo real del falso negativo es otro? (falso positivo = 1)
filas = []
for r in [1, 2, 3, 5, 10]:
    c_nunca, c_todos = tasa_fail * r, p_pass * 1
    filas.append({"FN : FP": f"{r} a 1",
                  "costo nunca avisar": round(c_nunca, 2), "costo avisar a todos": round(c_todos, 2),
                  "mejor sin modelo": "nunca avisar" if c_nunca < c_todos else "avisar a todos",
                  "avisar si P(Fail) ≥": round(1 / (1 + r), 3),
                  "NPV mínimo para superar a «avisar a todos»": f"{r / (r + 1):.1%}"})
display(pd.DataFrame(filas))
print(f"Punto de equilibrio entre las dos líneas base: {p_pass / tasa_fail:.2f} a 1 "
      f"(con el costo del FN por encima de {p_pass / tasa_fail:.2f} veces el del FP, avisar a todos es mejor que no avisar a nadie).")
print(f"Por cada 1.000 estudiantes: ≈ {1000 * tasa_fail:.0f} pierden el curso y ≈ {1000 * p_pass:.0f} aprueban.")

**Lo que dice la línea base (una frase):** con el supuesto 5 a 1, **no avisar a nadie** cuesta **3,46** por estudiante, el azar 1,28 y **avisar a todos solo 0,31**; esa última es la estrategia a vencer, porque con 69,2 % de reprobados «avisar a todos» solo falla con el 30,8 % que aprobaría.

**Qué tiene que lograr un modelo para valer la pena** (cada condición sale de las cifras de arriba):

1. **Costo por estudiante menor que 0,31.** Es equivalente a esta regla, que conviene recordar: *el modelo supera a «avisar a todos» solo si, entre los estudiantes a los que dice «tranquilo», al menos **5 de cada 6 (83,3 %) aprueban de verdad*** (el **NPV** de la sección 4). Con la proporción 5 a 1 cada falso negativo borra el beneficio de 5 estudiantes tranquilizados correctamente.
2. **Precisión mayor que 69,2 %** (la tasa base): si no, avisar según el modelo no mejora al azar.
3. **Menos de 100 alertas por cada 100 estudiantes**: si avisa a todos, no está priorizando nada.

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** **No.** «Avisar a todos» ya acierta 69,2 % sin distinguir a nadie, y «nunca avisar» acierta 30,8 % aun siendo lo peor. La exactitud tampoco refleja que un falso negativo cuesta 5 veces más que un falso positivo.

## 4. Las métricas explicadas

### 4.1 Cómo leer una alerta, en palabras simples

El modelo le pone a cada estudiante una etiqueta: **«avisar»** (riesgo de perder el curso) o **«tranquilo»**. Al compararla con lo que realmente pasó hay cuatro casos (VP, FN, FP y VN, de la sección 2).
Cada métrica es **una pregunta distinta** sobre esas cuatro casillas:

| Pregunta del negocio | Métrica que la responde |
|---|---|
| ¿Cuánto nos cuestan los errores, en total? | **Costo por estudiante** |
| De los que **sí pierden el curso**, ¿a cuántos avisamos? | ***Recall* de `Fail`** |
| De los que **avisamos**, ¿cuántos de verdad lo necesitaban? | **Precisión de `Fail`** |
| De los que dijimos **«tranquilo»**, ¿cuántos de verdad aprueban? | **NPV** (valor predictivo del «no avisar») |
| ¿Cuánto trabajo le genera esto a la institución? | **Alertas por 100** |
| Si solo hay tutoría para unos pocos, ¿estamos eligiendo a los correctos? | ***Recall* y precisión en el top k %** |
| ¿Trata el modelo igual a todos los grupos? | ***Recall* y precisión por grupo** |

### 4.2 Las métricas, una por una

| Métrica | Fórmula | En palabras simples | Cuándo mirarla | Cuidado |
|---|---|---|---|---|
| **Costo esperado por estudiante** | (5·FN + 1·FP) / n | «Cuánto pierde la institución, en promedio, por cada estudiante evaluado.» Pone los dos errores en la misma unidad | **Métrica principal de negocio** | Depende del supuesto 5 a 1: se reporta siempre junto con su sensibilidad |
| ***Recall* de `Fail`** (sensibilidad) | VP / (VP + FN) | «De los que realmente pierden el curso, ¿a qué fracción avisamos?» | **Principal**: el error caro es no avisar | Se «gana» avisando a todos (100 %): **no se lee sola**, siempre con la precisión |
| **Precisión de `Fail`** | VP / (VP + FP) | «Cuando el modelo avisa, ¿qué fracción acierta?» | Control: mide las alertas desperdiciadas | Con 69,2 % de `Fail`, **el azar ya da 69,2 %**: hay que superar esa tasa base |
| **NPV** (valor predictivo del «no avisar») | VN / (VN + FN) | «Cuando el modelo dice *tranquilo*, ¿qué fracción de verdad aprueba?» | **Clave en este caso**: debe superar 83,3 % (5 de cada 6) para vencer a «avisar a todos» | Si no se mira, un modelo puede avisar a casi todos y parecer bueno |
| **Especificidad** | VN / (VN + FP) | «De los que aprueban, ¿a cuántos dejamos tranquilos?» | Complementa al *recall*: mide el desperdicio de tutorías | Un modelo que avisa a todos tiene 0 % |
| **F1 y F2 de `Fail`** | F1 = 2PR/(P+R) · F2 = 5PR/(4P+R) | Un solo número que combina precisión y *recall* (**F2 pesa más el *recall***, acorde con el costo) | Para comparar modelos de un vistazo | «Avisar a todos» ya da F1 = 0,82: **necesita compararse con esa referencia** |
| **Exactitud balanceada** | (*recall* de `Fail` + especificidad) / 2 | Promedio de «detecto a los que pierden» y «dejo tranquilos a los que aprueban» | Control; no se deja engañar por la clase mayoritaria | Una estrategia sin modelo siempre da 50 % |
| **PR-AUC de `Fail`** | área bajo la curva precisión–*recall* | «¿Ordena bien a los estudiantes del más al menos riesgoso, sin elegir un umbral?» | Control del ranking; útil para priorizar con capacidad limitada | Se compara con la tasa base (69,2 %), no con 50 % |
| **Brier / calibración** | media de (probabilidad − resultado)² | «Si dice 80 % de riesgo, ¿de verdad pierden el curso 8 de cada 10?» | Necesaria para fijar el umbral por costos (avisar si P ≥ 1/(1+5) = 0,167) | Un modelo puede ordenar bien y estar mal calibrado |
| **Alertas por 100** | 100 · (VP + FP) / n | «¿A cuántos de cada 100 estudiantes avisamos?» | Operativa: compara con la capacidad real de tutoría | Avisar a 69 de cada 100 puede ser inviable |
| ***Recall* y precisión en el top k %** | VP entre el k % más riesgoso / total de `Fail` | «Si solo puedo ayudar al k % de los estudiantes, ¿cuántos de los que pierden el curso alcanzo?» | **Operativa**: define cuánto sirve el modelo con la capacidad real | La referencia del azar es k % de *recall* (con k = 40 %, un 40 %) |
| ***Recall* y precisión por grupo** | las mismas, por `gender`, `family_income`, `urban_rural`, `school_type` | «¿Falla más con algún grupo?» | **Equidad**, antes de dar el modelo por válido | Los grupos pequeños (`gender` = `Other`) tienen intervalos amplios |

**Métricas que NO sirven solas:**
- **Exactitud:** premia a la clase mayoritaria; «avisar a todos» ya acierta 69,2 %.
- ***Recall* solo:** se maximiza avisando a todos.
- **ROC-AUC y PR-AUC solas:** miden si el modelo *ordena* bien, pero no dicen cuánto cuesta ni dónde poner el umbral. Con 69,2 % de `Fail`, la PR-AUC parte de un valor alto (el azar da ≈ 0,69): se lee frente a esa referencia y junto con el costo y las métricas por capacidad.

In [ ]:
# Ejemplo REAL para entender cada métrica (NO es el modelo): una regla de una sola variable.
# «Avisar al 40 % de los estudiantes con peor nota previa (previous_exam_score)».
K_EJEMPLO = 40
riesgo = -obj.previous_exam_score.to_numpy()                    # más riesgo = peor nota previa
orden = np.argsort(-riesgo, kind="stable")
avisa = np.zeros(n, dtype=bool)
avisa[orden[: int(n * K_EJEMPLO / 100)]] = True
m = business_metrics(fail, avisa)

print(f"Regla de ejemplo: avisar al {K_EJEMPLO} % con peor nota previa  ->  VP={m['VP']:,}  FN={m['FN']:,}  FP={m['FP']:,}  VN={m['VN']:,}\n")
detalle = pd.DataFrame([
    ("Recall de Fail", f"VP / (VP + FN) = {m['VP']:,} / ({m['VP']:,} + {m['FN']:,})", m["recall Fail"], "Detecta a esa fracción de los que pierden el curso"),
    ("Precisión de Fail", f"VP / (VP + FP) = {m['VP']:,} / ({m['VP']:,} + {m['FP']:,})", m["precisión Fail"], f"De sus alertas, esa fracción lo necesitaba (la tasa base es {tasa_fail:.1%})"),
    ("NPV", f"VN / (VN + FN) = {m['VN']:,} / ({m['VN']:,} + {m['FN']:,})", m["NPV (confianza al tranquilizar)"], f"De los «tranquilos», esa fracción aprueba (hace falta ≥ {FALSE_NEGATIVE_COST / (FALSE_NEGATIVE_COST + 1):.1%})"),
    ("Especificidad", f"VN / (VN + FP) = {m['VN']:,} / ({m['VN']:,} + {m['FP']:,})", m["especificidad"], "Deja tranquilos a esa fracción de los que aprueban"),
    ("F1", "2·P·R / (P + R)", m["F1"], "Combina precisión y recall"),
    ("Exactitud balanceada", "(recall + especificidad) / 2", m["exactitud balanceada"], "Promedio de las dos clases"),
    ("Alertas por 100", f"100 · (VP + FP) / n", m["alertas por 100"], "Trabajo que genera para la institución"),
    ("Costo por estudiante", f"({FALSE_NEGATIVE_COST}·FN + {FALSE_POSITIVE_COST}·FP) / n", m["costo por estudiante"], f"Hay que bajar de {mejor_costo_base:.2f} (avisar a todos)"),
], columns=["métrica", "cálculo con los números del ejemplo", "valor", "cómo se lee"])
display(detalle.assign(valor=detalle["valor"].round(3)))

# Curva de capacidad: ¿qué pasa si solo se puede avisar al k % más riesgoso?
filas = []
for k in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    a = np.zeros(n, dtype=bool); a[orden[: int(n * k / 100)]] = True
    mk = business_metrics(fail, a)
    filas.append({"avisar al k %": k, "recall Fail": round(mk["recall Fail"], 3), "precisión Fail": round(mk["precisión Fail"], 3),
                  "recall / azar (k %)": round(mk["recall Fail"] / (k / 100), 2), "NPV": round(mk["NPV (confianza al tranquilizar)"], 3),
                  "costo por estudiante": round(mk["costo por estudiante"], 3)})
print("\nRegla de ejemplo con distinta capacidad (compara el costo con avisar a todos = %.3f):" % mejor_costo_base)
display(pd.DataFrame(filas))

**Lo que enseña el ejemplo (con una sola variable, que no es el modelo):**

- **Precisión alta, *recall* bajo:** al avisar solo al 40 % con peor nota previa, **85,8 %** de las alertas acierta (muy por encima de la tasa base de 69,2 %), pero solo se detecta a **49,6 %** de los que pierden el curso. Más de la mitad queda sin aviso: **34.900 falsos negativos**, el error caro.
- **El costo lo domina el *recall*, no la precisión:** cada falso negativo pesa 5; por eso bajar el costo exige aumentar el *recall*, y por eso hay que mirar las dos métricas juntas.
- **El NPV de ese 40 %** es bajo (41,8 %): entre los «tranquilos» más de la mitad **pierde el curso**. Por eso esta regla simple **no supera a «avisar a todos»** (costo 1,80 frente a 0,31).
- **Cómo leer la tabla de capacidad:** si solo se puede avisar al 40 %, un *recall* de 49,6 % rinde 1,24 veces más que el azar (que daría 40 %). Es la métrica que conecta al modelo con la realidad de la institución.
- **Para vencer a «avisar a todos»** el modelo tiene que **separar bien** a quienes aprobarán: subir mucho el NPV (≥ 83,3 %). Eso es lo que hay que buscar en las etapas de modelado.

## 5. Cómo se medirá el éxito

- **Métricas principales:** **costo esperado por estudiante**, ***recall* de `Fail`** y **NPV**.
- **Métricas de control:** precisión de `Fail`, F1 y F2, especificidad, exactitud balanceada, PR-AUC, calibración (Brier).
- **Métricas operativas:** alertas por 100 estudiantes y ***recall* / precisión en el top k %** (k según la capacidad de tutoría).
- **Métricas de equidad:** *recall* y precisión de `Fail` por `gender`, `family_income`, `urban_rural` y `school_type`.
- **Se mide en:** la partición de prueba, abierta **una sola vez** (etapa 7), con intervalo del 95 %; las decisiones se toman con validación.
- **El modelo «sirve» si, en la prueba:**
  1. su costo por estudiante es **menor que 0,31** (avisar a todos) y el intervalo de la diferencia queda por debajo de cero; **equivale a un NPV mayor que 83,3 %**;
  2. su precisión de `Fail` es **mayor que 69,2 %**;
  3. avisa a **menos de 100 de cada 100** estudiantes;
  4. cumple el *recall* mínimo y la capacidad de alertas que fije la institución (**TODO**, parámetros de la celda siguiente);
  5. no falla de forma desigual con ningún grupo (la brecha máxima aceptable la define el grupo: **TODO**).

In [ ]:
# Parámetros que debe fijar la INSTITUCIÓN (None = aún sin definir; no se inventan).
MIN_RECALL_FAIL = None        # p. ej. 0.90  → «detectar al menos 90 % de los que perderán el curso»
MAX_ALERTS_PER_100 = None     # p. ej. 50    → capacidad real de tutoría por cada 100 estudiantes
MAX_GROUP_RECALL_GAP = None   # p. ej. 0.05  → brecha máxima de recall entre grupos

NPV_MIN = FALSE_NEGATIVE_COST / (FALSE_NEGATIVE_COST + FALSE_POSITIVE_COST)    # 5/6 = 83,3 %


def meets_success(m, baseline_cost=mejor_costo_base, base_rate=tasa_fail):
    """Compara las métricas de un modelo (salida de business_metrics) con los criterios de éxito."""
    checks = {
        "costo menor que el de avisar a todos": m["costo por estudiante"] < baseline_cost,
        f"NPV mayor que {NPV_MIN:.1%} (equivale a vencer a «avisar a todos»)": m["NPV (confianza al tranquilizar)"] > NPV_MIN,
        "precisión mayor que la tasa base de Fail": m["precisión Fail"] > base_rate,
        "no avisa a todos (alertas < 100 por 100)": m["alertas por 100"] < 100,
    }
    if MIN_RECALL_FAIL is not None:
        checks[f"recall de Fail ≥ {MIN_RECALL_FAIL}"] = m["recall Fail"] >= MIN_RECALL_FAIL
    if MAX_ALERTS_PER_100 is not None:
        checks[f"alertas ≤ {MAX_ALERTS_PER_100} por 100"] = m["alertas por 100"] <= MAX_ALERTS_PER_100
    return checks


# Comprobación con las líneas base y con la regla de ejemplo (ninguna debería cumplir todo):
candidatos = {**{k: base.loc[k] for k in base.index}, f"Regla de ejemplo (top {K_EJEMPLO} %)": pd.Series(m)}
for nombre, serie in candidatos.items():
    resultado = meets_success(serie)
    print(f"{nombre}: {'cumple todo' if all(resultado.values()) else 'NO cumple'} -> " +
          ", ".join(f"{'✓' if v else '✗'} {k}" for k, v in resultado.items()))
if MAX_GROUP_RECALL_GAP is None:
    print("\nTODO: fijar MIN_RECALL_FAIL, MAX_ALERTS_PER_100 y MAX_GROUP_RECALL_GAP con la institución.")

## 6. Lo que podría salir mal

- **Fuga de información:** `exam_score` (de ella sale el objetivo: Fail si < 70), `performance_grade` y `performance_level` (derivadas de la nota), `questions_correct` (correlación 0,975 con la nota) y `questions_attempted` (dato del propio examen) **se excluyen**. `pass_status` (corte en 50) **tampoco se usa**: es otra versión de la misma respuesta. A medir **con y sin**: `time_management_score`, `exam_anxiety_level`, `exam_difficulty`, porque no se sabe si se conocen antes del examen.
- **Etiqueta definida por un corte:** el corte de 70 es una regla de la institución, y **21,8 % de los estudiantes está a menos de 5 puntos de él** (53,7 % pierde el curso). Pequeñas diferencias el día del examen cambian el resultado: el *recall* no llegará a 100 % y conviene reportar también el desempeño lejos del corte.
- **Mayoría de `Fail` (69,2 %):** «avisar a todos» es una línea base fuerte (costo 0,31). Se exige **NPV ≥ 83,3 %** y precisión sobre la tasa base. Hay que evitar que el modelo parezca bueno solo porque avisa a casi todos.
- **Capacidad:** no se puede apoyar al 69 % de los estudiantes; el modelo debe **ordenar por riesgo** y evaluarse con las métricas por capacidad (*recall* en el top k %).
- **Equidad:** posibles brechas por `gender`, `family_income`, `urban_rural`, `school_type` y `age` (hay menores). `gender` y `family_income` no entran como predictores; se audita el desempeño por grupo.
- **Datos personales:** sin nombres ni documentos; `student_id` es un seudónimo (se excluye). Licencia del dataset **pendiente de verificar** en Kaggle.
- **Alcance de la decisión:** el modelo **prioriza** quién recibe una revisión; una persona decide la intervención. No sanciona, excluye ni limita oportunidades, y no explica causas.
- **Límite de los datos:** el dataset parece sintético (varias variables casi no se relacionan con la nota) y no tiene fechas: no se evaluó estabilidad en el tiempo.

## 7. Cierre

- [ ] La regla de aprobación (`exam_score` ≥ 70) está escrita y comprobada, y `pass_status` (corte en 50) queda fuera.
- [ ] Costos escritos con su proporción, marcada como supuesto hasta validarla con la institución.
- [ ] Línea base calculada (nunca avisar, avisar a todos, azar) y entendida: la estrategia a vencer es «avisar a todos» (0,31).
- [ ] Métricas principales y de control elegidas, cada una con su motivo; se entiende por qué ninguna sirve sola.
- [ ] Parámetros `MIN_RECALL_FAIL`, `MAX_ALERTS_PER_100` y `MAX_GROUP_RECALL_GAP` definidos con la institución.
- [ ] Resumen copiado a `reports/00_caso.md`.
- [ ] Los `.md` del repositorio actualizados al corte de 70 (los notebooks `02b` y `03b`, `src/` y `tests/` ya lo están).
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué «avisar a todos» es la línea base a vencer y qué significa el NPV.